[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/18_Deduplication.ipynb)

# Semantica 中的去重

欢迎来到**去重(Deduplication)**演练!在任何知识图谱中,数据往往来自多个来源,从而产生重复的实体(例如 "Apple Inc." 与 "Apple Inc")。

Semantica 提供了强大的**去重模块**,帮助你:
1. **计算相似度**:使用字符串、属性和嵌入来比较实体。
2. **检测重复项**:找出代表同一个现实对象的实体对或实体组。
3. **聚类实体**:把相似的实体归为一组。
4. **合并实体**:将重复项合并为单一的规范实体,同时解决冲突。

本 Notebook 将通过清晰的示例带你走过每一步。

In [ ]:
# Install Semantica
!pip install -q semantica

## 1. 准备示例数据

让我们创建一个带有一些刻意重复项的数据集。我们将模拟来自不同来源(例如 CRM 和公共数据库)的数据。

**我们的实体:**
- **Apple**:变体包括 "Apple Inc."、"Apple Inc" 以及单独的 "Apple"。
- **Microsoft**:变体包括 "Microsoft Corp" 和 "Microsoft"。
- **Google**:作为对照的唯一实体。

In [ ]:
entities = [
    # Apple Variations
    {
        "id": "e1",
        "name": "Apple Inc.",
        "type": "Company",
        "properties": {"industry": "Technology", "hq": "Cupertino", "founded": 1976},
        "relationships": [{"predicate": "founded_by", "object": "Steve Jobs"}]
    },
    {
        "id": "e2",
        "name": "Apple Inc",
        "type": "Company",
        "properties": {"industry": "Tech", "hq": "Cupertino, CA"}, # Slightly different properties
        "relationships": []
    },
    {
        "id": "e3",
        "name": "Apple",
        "type": "Company",
        "properties": {"industry": "Consumer Electronics"}, 
        "relationships": [{"predicate": "ceo", "object": "Tim Cook"}]
    },
    
    # Microsoft Variations
    {
        "id": "e4",
        "name": "Microsoft Corp",
        "type": "Company",
        "properties": {"industry": "Software", "hq": "Redmond"}
    },
    {
        "id": "e5",
        "name": "Microsoft",
        "type": "Company",
        "properties": {"industry": "Tech", "hq": "Redmond, WA"}
    },
    
    # Unique Entity
    {
        "id": "e6",
        "name": "Google LLC",
        "type": "Company",
        "properties": {"industry": "Internet"}
    }
]

print(f"Created {len(entities)} sample entities.")

## 2. 相似度计算

`SimilarityCalculator` 是核心引擎。它比较两个实体并返回 0 到 1 之间的分数。它会考察:
- **字符串相似度**:名称和文本字段。
- **属性相似度**:键值对的重叠程度。
- **关系相似度**:与其他实体的连接。
- **嵌入**:语义向量相似度(如可用)。

你可以自定义每个因素的权重。

In [ ]:
from semantica.deduplication import SimilarityCalculator, SimilarityResult
# Initialize calculator with custom weights
calculator = SimilarityCalculator(
    string_weight=0.5,      # High importance on name
    property_weight=0.3,    # Medium importance on properties
    relationship_weight=0.2 # Lower importance on relationships
)

# Compare "Apple Inc." (e1) vs "Apple Inc" (e2)
score_e1_e2 = calculator.calculate_similarity(entities[0], entities[1])

print(f"Similarity between '{entities[0]['name']}' and '{entities[1]['name']}':")
print(f"  Total Score: {score_e1_e2.score:.4f}")
print(f"  Breakdown:   {score_e1_e2.components}")

# Compare "Apple Inc." (e1) vs "Microsoft" (e5)
score_e1_e5 = calculator.calculate_similarity(entities[0], entities[4])

print(f"\nSimilarity between '{entities[0]['name']}' and '{entities[4]['name']}':")
print(f"  Total Score: {score_e1_e5.score:.4f}")

## 3. 重复项检测

`DuplicateDetector` 使用相似度计算器扫描你的数据集以查找重复项。它可以找到:
- **实体对**:简单的 A 与 B 匹配。
- **实体组**:A 与 B 匹配,且 B 与 C 匹配。

它使用 `similarity_threshold`(相似度阈值)来判定什么算作匹配。

In [ ]:
# Import specific classes for Duplicate Detection
from semantica.deduplication import DuplicateDetector, DuplicateCandidate, DuplicateGroup
from semantica.deduplication import DeduplicationConfig

detector = DuplicateDetector(
    similarity_threshold=0.7, 
    confidence_threshold=0.6  
)

# Detect pairs
candidates = detector.detect_duplicates(entities)

print(f"Found {len(candidates)} duplicate pairs:")
for c in candidates:
    print(f"  - {c.entity1['name']} <==> {c.entity2['name']} (Score: {c.similarity_score:.2f})")

### 增量检测
如果你已有一个现有数据库并接入了新数据,就不想重新比较所有内容。请使用 `incremental_detect`。

In [ ]:
existing_db = entities[:3] # The Apple entities
new_data = [entities[4]]   # Microsoft
# Check if new data matches anything in existing DB
inc_candidates = detector.incremental_detect(new_data, existing_db)

print(f"New matches found: {len(inc_candidates)}")
# Expected: 0, because Microsoft is not Apple.

## 4. 聚类

有时仅有实体对是不够的。`ClusterBuilder` 会把相关的实体归入聚类。这有助于了解某个重复实体的完整范围。

In [ ]:
# Import specific classes for Clustering
from semantica.deduplication import ClusterBuilder, Cluster, ClusterResult
cluster_builder = ClusterBuilder(threshold=0.7)
result = cluster_builder.build_clusters(entities)

print(f"Found {len(result.clusters)} clusters:")
for i, cluster in enumerate(result.clusters):
    names = [e['name'] for e in cluster.entities]
    print(f"  Cluster {i+1}: {names}")

cluster_builder = ClusterBuilder(threshold=0.7)
result = cluster_builder.build_clusters(entities)

print(f"Found {len(result.clusters)} clusters:")
for i, cluster in enumerate(result.clusters):
    names = [e['name'] for e in cluster.entities]
    print(f"  Cluster {i+1}: {names}")

## 5. 实体合并

找到重复项之后,`EntityMerger` 会将它们合并。你需要选择一种**合并策略**:

- `KEEP_FIRST` / `KEEP_LAST`:基于顺序。
- `KEEP_MOST_COMPLETE`:保留数据最多的实体(属性 + 关系)。
- `KEEP_HIGHEST_CONFIDENCE`:使用内部置信度分数。
- `MERGE_ALL`:合并所有内容(数组会被拼接,冲突通过投票解决)。

In [ ]:
# Import specific classes for Entity Merging
from semantica.deduplication import EntityMerger, MergeStrategy, MergeStrategyManager, MergeOperation, MergeResult

In [ ]:
merger = EntityMerger()

# We will use the 'KEEP_MOST_COMPLETE' strategy
# This ensures we don't lose valuable information from richer entities
merge_ops = merger.merge_duplicates(
    entities, 
    strategy=MergeStrategy.KEEP_MOST_COMPLETE
)

print(f"Performed {len(merge_ops)} merge operations.")

print("\n--- Merged Results ---")
for op in merge_ops:
    final_ent = op.merged_entity
    original_count = len(op.source_entities)
    print(f"Merged {original_count} entities into: '{final_ent['name']}'")
    print(f"  - Final Properties: {final_ent['properties']}")
    print(f"  - Final Relationships: {len(final_ent.get('relationships', []))}")

## 6. 完整工作流

让我们把它封装成一个简洁的函数,输入脏数据,返回干净的数据。

In [ ]:
def deduplicate_dataset(raw_entities):
    print("1. Detecting duplicates...")
    # Step 1: Detect
    detector = DuplicateDetector(similarity_threshold=0.75)
    # We can skip explicit detection calls if we just want to merge, 
    # as EntityMerger calls detection internally, but doing it manually allows inspection.
    
    print("2. Merging entities...")
    # Step 2: Merge
    merger = EntityMerger()
    ops = merger.merge_duplicates(raw_entities, strategy=MergeStrategy.KEEP_MOST_COMPLETE)
    
    # Let's collect all final IDs to see what remains
    merged_entities = [op.merged_entity for op in ops]
    
    # Find entities that were NOT part of any merge (singletons)
    merged_ids = set()
    for op in ops:
        for source in op.source_entities:
            merged_ids.add(source['id'])
            
    singletons = [e for e in raw_entities if e['id'] not in merged_ids]
    
    final_dataset = merged_entities + singletons
    return final_dataset

# Run the workflow
clean_data = deduplicate_dataset(entities)

print(f"\nOriginal Size: {len(entities)}")
print(f"Cleaned Size:  {len(clean_data)}")
print("\nFinal Entity Names:")
for e in clean_data:
    print(f"  - {e['name']}")

## 总结

你已经学会如何:
1. **导入**所需的去重类。
2. 在实体之间**计算相似度**。
3. 使用可配置的阈值**检测重复项**。
4. 对相似实体进行**聚类**。
5. 将重复项**合并**成干净、规范的数据集。

该模块对于维护高质量的知识图谱至关重要,尤其是在接入来自多个、可能杂乱的来源的数据时。